# AdaptiveHop — QA evaluation on a GPU-hosted model

Runs the Task 3 baseline against an open model loaded on a **free Colab or
Kaggle GPU**, instead of a rate-limited hosted API.

**Why:** Groq's free tier allows 200k tokens/day and Gemini's allows 20
requests/day per model, so a 300-question sweep takes days. A T4 runs the
whole k = 3 / 5 / 10 sweep in roughly an hour, with no quota and no daily
reset — so it can be repeated every time the retriever changes.

**Before running:** enable the GPU.

- Colab: *Runtime -> Change runtime type -> T4 GPU*
- Kaggle: *Settings -> Accelerator -> GPU T4 x2*, and *Internet: On*


## 1. Check the GPU


In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), 'No GPU. Enable the accelerator (see above) and restart.'
print('GPU :', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GB')


## 2. Get the code

Set `REPO_URL` to your fork. If the repository is private, use a token URL
(`https://<token>@github.com/...`) or upload a zip and skip this cell.

Re-running this cell **hard-resets an existing checkout to the pushed tip**.
Colab sessions keep `/content` between runs, so without that an old clone
silently persists and you debug code that is no longer on the branch. Any
results you have not downloaded yet should be saved first (section 8).


In [ ]:
REPO_URL = 'https://github.com/GUEST72/Adaptive-Evidence-Retrieval-for-Multi-Hop-Question-Answering.git'
BRANCH   = 'feature/qa-baseline'
REPO_DIR = 'Adaptive-Evidence-Retrieval-for-Multi-Hop-Question-Answering'

import os, subprocess

def git(*args, cwd=None):
    subprocess.run(['git', *args], cwd=cwd, check=True)

if os.path.isdir(REPO_DIR):
    # Existing checkout: bring it to the pushed tip rather than trusting it.
    git('fetch', '--depth', '1', 'origin', BRANCH, cwd=REPO_DIR)
    git('reset', '--hard', f'origin/{BRANCH}', cwd=REPO_DIR)
else:
    git('clone', '--branch', BRANCH, '--depth', '1', REPO_URL, REPO_DIR)

os.chdir(REPO_DIR)
print('working dir:', os.getcwd())
!git log --oneline -1

# Fail loudly here rather than with a cryptic SyntaxError ten cells later.
import glob
for path in glob.glob('baseline/*.py') + glob.glob('scripts/*.py') + glob.glob('evaluation/*.py'):
    head = open(path, encoding='utf-8').read()
    if '<<<<<<<' in head or '>>>>>>>' in head:
        raise RuntimeError(f'{path} contains merge conflict markers — checkout is broken or stale.')
print('checkout OK: no conflict markers')


In [ ]:
!pip install -q pyyaml python-dotenv httpx
!pip install -q 'transformers>=4.44' accelerate bitsandbytes


## 3. Get the dataset

MuSiQue-Ans is not versioned in the repo (see `data/musique_ans/README.md`).
Uncomment whichever route is easiest. The loader accepts either the short
(`dev.jsonl`) or official (`musique_ans_v1.0_dev.jsonl`) filename.


In [ ]:
import os, shutil
os.makedirs('data/musique_ans', exist_ok=True)
TARGET = 'data/musique_ans/musique_ans_v1.0_dev.jsonl'

# --- Option A: Google Drive (Colab) ---
# from google.colab import drive; drive.mount('/content/drive')
# shutil.copy('/content/drive/MyDrive/musique_ans_v1.0_dev.jsonl', TARGET)

# --- Option B: upload from your machine (Colab) ---
# from google.colab import files
# uploaded = files.upload()
# shutil.move(next(iter(uploaded)), TARGET)

# --- Option C: a Kaggle dataset attached to this notebook ---
# shutil.copy('/kaggle/input/<your-dataset>/musique_ans_v1.0_dev.jsonl', TARGET)

assert os.path.exists(TARGET), 'Uncomment one option above to supply the dev split.'
print('dev split:', round(os.path.getsize(TARGET)/1024**2, 1), 'MB')


Validating with Task 1's own loader — the expected hop counts confirm the
file is the right one and is intact.


In [ ]:
import sys; sys.path.insert(0, os.getcwd())
from collections import Counter
from src.data.musique_loader import load_split

records = load_split('dev')
print('records :', len(records))
print('hop mix :', dict(sorted(Counter(r.hop_count for r in records).items())))
print('expected: {2: 1252, 3: 760, 4: 405}')


## 4. Retrieval quality — free, no model needed

Worth running first: it needs no GPU and no API, and it is the number that
actually bounds EM. A question is only answerable if *every* supporting
paragraph is retrieved.


In [ ]:
!python scripts/run_retrieval_eval.py --config configs/baseline_gpu.yaml


## 5. Load the model

`Qwen2.5-7B-Instruct` in 4-bit fits a 16 GB T4 comfortably. On an
out-of-memory error, set `MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'` and
`LOAD_IN_4BIT = False`.


In [ ]:
MODEL_ID     = 'Qwen/Qwen2.5-7B-Instruct'
LOAD_IN_4BIT = True

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

quantization = None
if LOAD_IN_4BIT:
    quantization = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type='nf4',
    )

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization,
    torch_dtype=torch.float16,
    device_map='auto',
)
model.eval()
print('loaded', MODEL_ID)


## 6. Register it as a provider

`baseline/providers.py` exposes `register_provider`, so this notebook plugs a
GPU backend into the same interface Groq and Gemini use. Nothing in the
pipeline changes, and `torch` never becomes a dependency of the repo.

The prompt goes through the model's own chat template as a single user
message, matching how the hosted providers send it; `temperature=0` maps to
greedy decoding.


In [ ]:
import torch
from baseline import providers

@torch.inference_mode()
def hf_local_complete(prompt: str, model_name: str, max_tokens: int, temperature: float) -> str:
    messages = [{'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=temperature > 0,
        temperature=temperature if temperature > 0 else None,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated = output[0][inputs['input_ids'].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

providers.register_provider('hf_local', hf_local_complete)
print('registered. providers now:', sorted(providers.PROVIDERS))


A smoke test before spending an hour on the sweep.


In [ ]:
from baseline.qa_pipeline import _build_prompt

evidence = [
    {'idx': 3, 'title': 'Bauhaus', 'text': 'The Bauhaus was founded in 1919 by Walter Gropius in Weimar.', 'score': 1.0},
    {'idx': 7, 'title': 'Weimar', 'text': 'Weimar is a city in Thuringia, Germany.', 'score': 0.4},
]
print(repr(hf_local_complete(_build_prompt('Who founded the Bauhaus?', evidence), MODEL_ID, 512, 0.0)))
# expected: 'Walter Gropius'


## 7. Run the sweep

Called **in-process**, not through `!python`: a subprocess would not inherit
the `hf_local` provider registered above.

The sample is drawn once and reused for every k, so the three runs are
directly comparable. Each k writes `baseline/results/predictions_k{k}.jsonl`
as it goes, and responses are cached in `.cache/llm/responses.db`, so an
interrupted run resumes without recomputing.


In [ ]:
import time, yaml, pathlib
from baseline.runner import run_baseline, print_report, select_records
from baseline.placeholder_retriever import retrieve as placeholder_retrieve

config = yaml.safe_load(pathlib.Path('configs/baseline_gpu.yaml').read_text())
config['model'] = MODEL_ID

records = select_records(config)   # drawn once, shared across every k
print('evaluation sample:', len(records), 'questions')

outcomes = {}
for k in (3, 5, 10):
    config['k'] = k
    print(f'===== k={k} =====')
    started = time.time()
    outcome = run_baseline(config, retrieve=placeholder_retrieve, records=records)
    print_report(outcome)
    print(f'({time.time() - started:.0f}s)\n')
    outcomes[k] = outcome


## 8. Collect the results

Scores every prediction file present and prints one table. Download
`baseline/results/` afterwards and commit it to the repo.


In [ ]:
import json, glob, re
from baseline.qa_pipeline import QAResult
from evaluation.qa_eval import evaluate

paths = sorted(glob.glob('baseline/results/predictions_k*.jsonl'),
               key=lambda p: int(re.search(r'k(\d+)', p).group(1)))

print(f"{'k':>3} {'hops':>5} {'n':>5} {'EM':>7} {'F1':>7}")
for path in paths:
    k = int(re.search(r'k(\d+)', path).group(1))
    rows = [json.loads(line) for line in open(path) if line.strip()]
    if not rows:
        continue
    results = [QAResult(question_id=r['question_id'], hop_count=r['hop_count'],
                        predicted_answer=r['predicted_answer'], gold_answer=r['gold_answer'],
                        gold_aliases=tuple(r['gold_aliases']),
                        retrieved_indices=tuple(r['retrieved_indices'])) for r in rows]
    report = evaluate(results)
    o = report.overall
    print(f'{k:>3} {"all":>5} {o.count:>5} {o.em:>7.3f} {o.f1:>7.3f}')
    for hop, m in report.by_hop.items():
        print(f'{"":>3} {hop:>5} {m.count:>5} {m.em:>7.3f} {m.f1:>7.3f}')


Colab: download the results. On Kaggle they are already in `/kaggle/working`.


In [ ]:
# import shutil
# from google.colab import files
# shutil.make_archive('results', 'zip', 'baseline/results')
# files.download('results.zip')
